# KuCoin Data Downloader

### Objective:
- Download and save historical price data from the KuCoin cryptocurrency exchange as CSV files.

In [5]:
# Collecting all imports in one place
import pandas as pd
import requests
import time
import random
import re
import zipfile
import os 

## Function for data downloading

In [9]:
#Функция для загрузки данных с сайта kucoin
def download_file(data_type='klines',      #На выбор: klines, trades.
                  timeframe=None,          #Для klines: 1m, 5m, 15m, 1h, 8h, 12h, 1d,. 
                  date='2024-01-01',       #Формат 2020-08-08
                  ticker='BTCUSDT'):
    
    #Форматируем так итоговую строку если тип данных свечи
    if 'klines' in data_type.lower():
        url = f"https://historical-data.kucoin.com/data/spot/daily/{data_type}/{ticker}/{timeframe}/{ticker}-{timeframe}-{date}.zip"
    #Во всех остальных случаях так
    else:
        url = f"https://historical-data.kucoin.com/data/spot/daily/{data_type}/{ticker}/{ticker}-trades-{date}.zip"
    print(url)
    
    #Директория для сохранения файла
    directory = f"/Users/danielschollenberg/Documents/Трейдинг/Данные/kucoin/spot/{data_type}/{ticker}"
    #Создаем директорию, если она не существует
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Директория {directory} создана.")
            
    #Обход возможных блокировок:
    #Добавляем случайную задержку от 0 до 2000 миллисекунд (2 секунд) перед запросом
    delay = random.uniform(0, 2)
    print(f"Задержка перед запросом: {delay:.2f} секунд")
    time.sleep(delay)
    #Список заголовков User-Agent
    user_agents = ['Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
                   'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.1 Safari/605.1.15',
                   'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0',
                   'Mozilla/5.0 (iPhone; CPU iPhone OS 14_6 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Mobile/15E148 Safari/604.1',
                   'Mozilla/5.0 (Linux; Android 10; SM-G975F) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.105 Mobile Safari/537.36']
    #Выбираем случайный заголовок User-Agent
    headers = {'User-Agent': random.choice(user_agents)}
        
    #Создаем запрос получение данных в потоковом режиме
    response = requests.get(url, headers=headers, stream=True)
    #Если подключение прошло успешно - получен код 200 качаем и записываем на диск файл
    if response.status_code == 200:
        filename = directory + f"/{ticker}_{date}.zip"
        with open(filename, 'wb') as f:
            #Записываем весь контент сразу в файл
            f.write(response.content)  
        print(f"Файл {filename} успешно скачан, сохранен на диск")
    else:
        print(f"Ошибка при скачивании файла: статус {response.status_code}, URL: {url}")
        
    #Распаковывает zip-файл в указанную директорию  
    def unpack_zip(zip_path, extract_to):
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_to)
            #Удаление zip-файла после распаковки
            os.remove(zip_path)
            print(f"Архив {zip_path} удален после распаковки.")
        except zipfile.BadZipFile:
            print("Ошибка при распаковке: файл не является zip-архивом")
    try:
        
        #Распакуем файл
        unpack_zip(filename, os.path.dirname(filename))
        print(f"Файл {filename} успешно распакован.")
    except:
        print('Ошибка при распаковке')

In [11]:
# Example of usage
download_file(data_type='trades', date='2024-01-01')

https://historical-data.kucoin.com/data/spot/daily/trades/BTCUSDT/BTCUSDT-trades-2024-01-01.zip
Директория /Users/danielschollenberg/Documents/Трейдинг/Данные/kucoin/spot/trades/BTCUSDT создана.
Файл /Users/danielschollenberg/Documents/Трейдинг/Данные/kucoin/spot/trades/BTCUSDT/BTCUSDT_2024-01-01.zip успешно скачан, сохранен на диск
Архив /Users/danielschollenberg/Documents/Трейдинг/Данные/kucoin/spot/trades/BTCUSDT/BTCUSDT_2024-01-01.zip удален после распаковки.
Файл /Users/danielschollenberg/Documents/Трейдинг/Данные/kucoin/spot/trades/BTCUSDT/BTCUSDT_2024-01-01.zip успешно распакован.
